# Course Material Tutor

## Turning Lecture Notes into an AI Study Companion — Q&A, Summaries, and Quiz Mode

**Project 20 / 20** — GenAI Learning Series

---

| Property | Value |
|----------|-------|
| Task | Educational RAG — course-material retrieval with Q&A, summarisation, and auto-quiz |
| Architecture | Heading-aware chunking → TF-IDF / embedding index → multi-mode tutor pipeline |
| Modes | **Q&A** (answer questions from notes), **Summary** (condense a topic), **Quiz** (auto-generate MCQs) |
| Corpus | 6 embedded lecture-note documents (~50 sections) covering an introductory ML course |
| Evaluation | Retrieval recall, answer grounding, quiz quality — quantitative + qualitative |


## Project Overview

Students drown in slides, PDFs, and handouts. A **Course Material Tutor** indexes
all lecture content and exposes three study modes:

1. **Q&A mode** — ask a question, get a grounded answer with citations from the notes.
2. **Summary mode** — request a concise summary of any topic in the syllabus.
3. **Quiz mode** — auto-generate multiple-choice questions (MCQs) to self-test.

This notebook builds that tutor from scratch using only local, reproducible
components — no API keys required. It demonstrates **educational RAG design**:
how to chunk academic material, boost retrieval with section metadata, and
post-process retrieved passages into different output formats.


## Learning Goals

After completing this notebook you will understand:

1. How to parse structured lecture notes (markdown headings, bullet lists) into
   section-level chunks that preserve pedagogical context.
2. How to build a retrieval index over academic content with TF-IDF (baseline)
   and optionally sentence-transformer embeddings.
3. How to implement three distinct downstream modes (Q&A, summary, quiz) on top
   of the same retrieval engine.
4. How metadata boosting (topic tags, difficulty level, section depth) improves
   retrieval relevance for educational queries.
5. How to evaluate an educational RAG system: retrieval recall, answer grounding,
   and quiz question quality.


## Problem Statement

An introductory Machine Learning course has **6 lecture modules**, each with
structured notes covering theory, examples, and key formulas. Students need:

- Quick answers to concept questions ("What is the bias-variance tradeoff?")
- On-demand summaries before exams ("Summarise regularisation techniques")
- Self-test quizzes generated from the actual lecture content

**Goal:** Build a local Course Material Tutor that retrieves relevant sections
from the lecture corpus and formats responses for each of the three study modes.


## Why This Project Matters

| Stakeholder | Benefit |
|-------------|---------|
| **Students** | Instant, citation-backed answers from their own course material |
| **Instructors** | Auto-generated quiz banks aligned to lecture content |
| **Course designers** | Gap analysis — which topics have weak coverage? |
| **EdTech platforms** | Template for building RAG tutors over any curriculum |

Educational RAG is different from generic document QA because:
- **Chunking must respect pedagogical structure** (topics → sub-topics → examples)
- **Retrieval must balance breadth** (cover prerequisites) and **depth** (specific answer)
- **Output format varies by mode** (prose answer vs bullet summary vs MCQ)


## Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pkgs = {
    "scikit-learn": "sklearn",
    "numpy": "numpy",
}

for pip_name, import_name in pkgs.items():
    try:
        __import__(import_name)
    except ImportError:
        _install(pip_name)

# Optional: sentence-transformers for embedding-based retrieval
USE_ST = False
try:
    from sentence_transformers import SentenceTransformer
    USE_ST = True
    print("sentence-transformers available — will use embedding retrieval")
except ImportError:
    print("sentence-transformers not installed — using TF-IDF retrieval (fully functional)")

print("Environment ready.")


## Imports

In [ ]:
import os, re, json, random, textwrap, warnings
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from collections import defaultdict

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports loaded.")


## Configuration

In [ ]:
# ── Retrieval config ──
TOP_K          = 5        # sections retrieved per query
MIN_CHUNK_LEN  = 30       # discard very short sections (characters)

# ── Quiz config ──
QUIZ_DISTRACTORS = 3      # wrong options per MCQ

# ── Embedding model (optional) ──
EMBED_MODEL = "all-MiniLM-L6-v2"

print("Configuration set.")


## Lecture Note Corpus

We embed **6 lecture modules** from an introductory Machine Learning course.
Each module is structured with headings, sub-headings, bullet points, and
inline examples — mimicking real lecture slides converted to markdown.

> In production, you would load `.md`, `.txt`, or parsed `.pdf` / `.pptx` files
> from a course folder. Here we embed realistic content directly in the notebook
> for full reproducibility.


In [ ]:
LECTURE_NOTES = [
    {
        "module_id": "ML01",
        "title": "Introduction to Machine Learning",
        "tags": ["ml-basics", "supervised", "unsupervised", "terminology"],
        "difficulty": "beginner",
        "content": """# Introduction to Machine Learning

## What is Machine Learning?

Machine learning (ML) is a subfield of artificial intelligence that gives computers
the ability to learn from data without being explicitly programmed. Instead of
writing rules by hand, we let algorithms discover patterns in data.

**Key idea:** Given a dataset of examples, an ML model learns a function that maps
inputs to outputs and generalises to unseen data.

## Types of Machine Learning

### Supervised Learning
In supervised learning, the training data includes both input features and the
correct output (label). The model learns to predict the label for new inputs.

- **Classification:** Predict a discrete category (spam vs not-spam, cat vs dog).
- **Regression:** Predict a continuous value (house price, temperature).

Examples: logistic regression, decision trees, SVMs, neural networks.

### Unsupervised Learning
In unsupervised learning, the training data has no labels. The model discovers
hidden structure in the data.

- **Clustering:** Group similar data points (K-means, DBSCAN).
- **Dimensionality reduction:** Compress high-dimensional data (PCA, t-SNE).

### Reinforcement Learning
An agent interacts with an environment, taking actions and receiving rewards.
The goal is to learn a policy that maximises cumulative reward.

## The ML Pipeline

1. **Data collection** — gather raw data from sensors, databases, APIs.
2. **Preprocessing** — clean, normalise, handle missing values.
3. **Feature engineering** — create informative features from raw data.
4. **Model selection** — choose a model family (tree, linear, neural).
5. **Training** — fit the model on training data.
6. **Evaluation** — measure performance on held-out test data.
7. **Deployment** — serve the model in production.

## Key Terminology

| Term | Meaning |
|------|---------|
| Feature | A measurable property of a data point (input variable) |
| Label / Target | The output the model predicts |
| Training set | Data used to fit the model |
| Test set | Data used to evaluate generalisation |
| Overfitting | Model memorises training data, fails on new data |
| Underfitting | Model is too simple to capture patterns |
"""
    },
    {
        "module_id": "ML02",
        "title": "Linear Models and Regression",
        "tags": ["linear-regression", "gradient-descent", "regularisation", "OLS"],
        "difficulty": "beginner",
        "content": """# Linear Models and Regression

## Simple Linear Regression

Simple linear regression models the relationship between one feature x and a
continuous target y as a straight line:

    y = w * x + b

where w is the slope (weight) and b is the intercept (bias).

**Goal:** Find w and b that minimise the sum of squared errors (SSE):

    SSE = sum((y_i - (w * x_i + b))^2)

This is called Ordinary Least Squares (OLS).

## Multiple Linear Regression

When we have multiple features x1, x2, ..., xp:

    y = w1*x1 + w2*x2 + ... + wp*xp + b

In matrix form: y = X @ w + b

The OLS solution is: w = (X^T X)^(-1) X^T y

## Gradient Descent

When the dataset is large, computing the exact OLS solution is expensive.
Gradient descent is an iterative optimisation algorithm:

1. Initialise weights randomly.
2. Compute the gradient of the loss w.r.t. each weight.
3. Update: w = w - learning_rate * gradient
4. Repeat until convergence.

**Variants:**
- **Batch gradient descent:** uses the full dataset per update.
- **Stochastic gradient descent (SGD):** uses one sample per update.
- **Mini-batch SGD:** uses a small batch (32–256 samples) per update.

The **learning rate** controls step size. Too large → divergence. Too small → slow.

## Regularisation

Regularisation adds a penalty to the loss function to prevent overfitting.

### L2 Regularisation (Ridge)
    Loss = SSE + lambda * sum(w_i^2)

Shrinks weights toward zero but rarely sets them exactly to zero.

### L1 Regularisation (Lasso)
    Loss = SSE + lambda * sum(|w_i|)

Can drive some weights to exactly zero → automatic feature selection.

### Elastic Net
    Loss = SSE + lambda1 * sum(|w_i|) + lambda2 * sum(w_i^2)

Combines L1 and L2. Useful when features are correlated.

## Evaluating Regression Models

| Metric | Formula | Notes |
|--------|---------|-------|
| MSE | mean((y - y_hat)^2) | Penalises large errors more |
| RMSE | sqrt(MSE) | Same units as y |
| MAE | mean(|y - y_hat|) | Robust to outliers |
| R^2 | 1 - SSE/SST | Proportion of variance explained |
"""
    },
    {
        "module_id": "ML03",
        "title": "Classification and Decision Boundaries",
        "tags": ["classification", "logistic-regression", "decision-trees", "metrics"],
        "difficulty": "intermediate",
        "content": """# Classification and Decision Boundaries

## Logistic Regression

Despite the name, logistic regression is a **classification** algorithm.
It models the probability that an input belongs to class 1:

    P(y=1|x) = sigmoid(w^T x + b)
    sigmoid(z) = 1 / (1 + exp(-z))

The output is always between 0 and 1. We classify as 1 if P >= 0.5.

**Loss function:** Binary cross-entropy (log loss):

    L = -[y * log(p) + (1-y) * log(1-p)]

## Decision Trees

A decision tree splits the feature space into regions using axis-aligned splits.

**Splitting criteria:**
- **Gini impurity:** measures how often a randomly chosen element would be
  misclassified. Gini = 1 - sum(p_i^2).
- **Information gain (entropy):** measures reduction in uncertainty.
  Entropy = -sum(p_i * log2(p_i)).

**Advantages:** Interpretable, handles non-linear boundaries, no scaling needed.
**Disadvantages:** Prone to overfitting, unstable (small data changes → different tree).

## Support Vector Machines (SVM)

SVMs find the hyperplane that maximises the **margin** — the distance between the
decision boundary and the nearest data points (support vectors).

**Kernel trick:** Maps data to a higher-dimensional space where a linear separator
exists. Common kernels: linear, RBF, polynomial.

## Classification Metrics

| Metric | What it measures |
|--------|-----------------|
| Accuracy | (TP + TN) / total — misleading if classes are imbalanced |
| Precision | TP / (TP + FP) — of predicted positives, how many are correct? |
| Recall | TP / (TP + FN) — of actual positives, how many were found? |
| F1 score | 2 * (precision * recall) / (precision + recall) — harmonic mean |
| ROC-AUC | Area under the ROC curve — threshold-independent ranking quality |
| Confusion matrix | 2x2 table showing TP, FP, FN, TN counts |

**When to use which:**
- Balanced data → accuracy is fine.
- Imbalanced data → focus on precision, recall, F1, PR-AUC.
- Spam detection → high precision (avoid false positives in inbox).
- Medical diagnosis → high recall (don't miss sick patients).
"""
    },
    {
        "module_id": "ML04",
        "title": "Ensemble Methods and Boosting",
        "tags": ["random-forest", "bagging", "boosting", "XGBoost", "ensemble"],
        "difficulty": "intermediate",
        "content": """# Ensemble Methods and Boosting

## Why Ensembles?

A single model may overfit or underfit. Ensembles combine multiple models to
reduce variance, reduce bias, or both.

**Key insight:** A committee of diverse weak learners can outperform a single
strong learner — if their errors are uncorrelated.

## Bagging (Bootstrap Aggregating)

1. Draw N bootstrap samples (random subsets with replacement) from the training data.
2. Train one model on each bootstrap sample.
3. Combine predictions: majority vote (classification) or average (regression).

**Effect:** Reduces variance. Works best with high-variance models (deep trees).

### Random Forest

A random forest is bagging applied to decision trees with an additional twist:
at each split, only a random subset of features is considered.

- Reduces correlation between trees → more diversity → better ensemble.
- Typically sqrt(p) features for classification, p/3 for regression.
- Out-of-bag (OOB) error: free validation using samples not in each bootstrap.

## Boosting

Boosting trains models **sequentially**, each one correcting the errors of the
previous ensemble.

### AdaBoost
1. Train a weak learner (e.g., decision stump).
2. Increase weights of misclassified samples.
3. Train next learner on the reweighted data.
4. Combine with weighted vote.

### Gradient Boosting
1. Fit an initial model (e.g., mean of y).
2. Compute residuals (errors).
3. Fit a new tree to the residuals.
4. Update predictions: y_hat = y_hat + learning_rate * new_tree(x).
5. Repeat.

**Key hyperparameters:**
- `n_estimators`: number of trees.
- `learning_rate`: shrinkage per step (smaller → more trees needed → better).
- `max_depth`: depth per tree (shallow trees for boosting, typically 3–8).

### XGBoost
Extreme Gradient Boosting — an optimised implementation with:
- Regularised objective (L1/L2 on leaf weights).
- Second-order gradient approximation (Newton step).
- Column subsampling and histogram-based splits for speed.
- Built-in handling of missing values.

## Bagging vs Boosting

| Property | Bagging | Boosting |
|----------|---------|----------|
| Training | Parallel | Sequential |
| Focus | Reduce variance | Reduce bias |
| Base learners | Independent | Dependent |
| Risk | Low overfitting | Can overfit if too many rounds |
"""
    },
    {
        "module_id": "ML05",
        "title": "Model Selection and Cross-Validation",
        "tags": ["cross-validation", "hyperparameter-tuning", "bias-variance", "model-selection"],
        "difficulty": "intermediate",
        "content": """# Model Selection and Cross-Validation

## The Bias-Variance Tradeoff

**Bias:** Error from wrong assumptions in the model (underfitting).
**Variance:** Error from sensitivity to training data fluctuations (overfitting).

    Total Error = Bias^2 + Variance + Irreducible Noise

- High bias → model too simple → underfitting (both train and test error high).
- High variance → model too complex → overfitting (train error low, test error high).
- Sweet spot → balance bias and variance for minimum total error.

## Train / Validation / Test Split

- **Training set (60-70%):** Fit the model.
- **Validation set (15-20%):** Tune hyperparameters, select model.
- **Test set (15-20%):** Final unbiased estimate of performance.

**Critical rule:** Never use the test set for model selection or tuning.
The test set must only be used once — at the very end.

## Cross-Validation

### K-Fold Cross-Validation
1. Split data into K equal folds (typically K=5 or 10).
2. For each fold: train on K-1 folds, validate on the remaining fold.
3. Average the K validation scores.

**Benefits:**
- Uses all data for both training and validation.
- More reliable estimate than a single train/val split.
- Reduces variance of the performance estimate.

### Stratified K-Fold
Ensures each fold has the same class distribution as the full dataset.
Essential for imbalanced classification problems.

### Leave-One-Out (LOO)
K = N (each sample is a fold). Very expensive but gives the least biased estimate.
Useful for very small datasets.

## Hyperparameter Tuning

### Grid Search
Exhaustively evaluates all combinations of a predefined hyperparameter grid.
- Simple but expensive: O(n1 * n2 * ... * nk) evaluations.

### Random Search
Randomly samples hyperparameter combinations. More efficient than grid search
when only a few hyperparameters matter (Bergstra & Bengio, 2012).

### Bayesian Optimisation
Uses a surrogate model (e.g., Gaussian Process) to predict which hyperparameter
regions are promising. Explores promising areas, exploits known good areas.
Efficient for expensive evaluations.

## Practical Model Selection Workflow

1. Start with a simple baseline (e.g., logistic regression, mean predictor).
2. Try a moderate model (random forest, gradient boosting).
3. Tune top 2-3 models with cross-validated random search.
4. Compare on validation set using the primary metric.
5. Evaluate the best model on the test set — report honestly.
"""
    },
    {
        "module_id": "ML06",
        "title": "Neural Networks and Deep Learning Foundations",
        "tags": ["neural-networks", "backpropagation", "activation-functions", "deep-learning"],
        "difficulty": "advanced",
        "content": """# Neural Networks and Deep Learning Foundations

## The Perceptron

The simplest neural network: a single neuron that computes a weighted sum of
inputs and applies a step function.

    output = step(w1*x1 + w2*x2 + ... + b)

Limitation: can only learn linearly separable functions (cannot solve XOR).

## Multi-Layer Perceptrons (MLPs)

Stacking layers of neurons creates a multi-layer perceptron (feedforward network):

    Input layer → Hidden layer(s) → Output layer

Each layer applies: output = activation(W @ input + b)

**Universal approximation theorem:** An MLP with one hidden layer and enough neurons
can approximate any continuous function — but may require impractically many neurons.

## Activation Functions

| Function | Formula | Properties |
|----------|---------|------------|
| Sigmoid | 1/(1+exp(-z)) | Output [0,1], vanishing gradient problem |
| Tanh | (exp(z)-exp(-z))/(exp(z)+exp(-z)) | Output [-1,1], zero-centred |
| ReLU | max(0, z) | Fast, no vanishing gradient, but "dying ReLU" |
| Leaky ReLU | max(0.01z, z) | Fixes dying ReLU |
| GELU | z * CDF(z) | Smooth, used in transformers |

**ReLU** is the default for hidden layers. **Sigmoid** for binary output.
**Softmax** for multi-class output (outputs sum to 1).

## Backpropagation

Backpropagation computes the gradient of the loss w.r.t. each weight using the
chain rule, moving backward from the output layer to the input layer.

**Steps:**
1. Forward pass: compute predictions.
2. Compute loss.
3. Backward pass: compute gradients via chain rule.
4. Update weights: w = w - lr * gradient.

**Common issues:**
- Vanishing gradients: gradients → 0 in deep networks (use ReLU, skip connections).
- Exploding gradients: gradients → ∞ (use gradient clipping, batch normalisation).

## Practical Deep Learning Tips

- **Batch normalisation:** normalise activations per mini-batch → faster training.
- **Dropout:** randomly zero out neurons during training → regularisation.
- **Learning rate scheduling:** start high, decay over time (cosine, step).
- **Weight initialisation:** Xavier (sigmoid/tanh), He (ReLU) — avoid starting at zero.
- **Data augmentation:** artificially expand training data (flips, rotations, crops).
- **Early stopping:** stop training when validation loss stops improving.

## CNNs vs RNNs (Preview)

- **CNNs (Convolutional Neural Networks):** spatial feature extraction via learnable
  filters. Used for images, audio spectrograms.
- **RNNs (Recurrent Neural Networks):** sequential processing with memory.
  Used for time series, text. LSTM and GRU solve the vanishing gradient problem.
- **Transformers:** attention-based, parallel processing. Now dominant in NLP and
  increasingly in vision (ViT, DINO).
"""
    },
]

print(f"Loaded {len(LECTURE_NOTES)} lecture modules:")
for lec in LECTURE_NOTES:
    lines = lec["content"].strip().splitlines()
    print(f"  [{lec['module_id']}] {lec['title']} — {len(lines)} lines, "
          f"difficulty: {lec['difficulty']}, tags: {lec['tags']}")


## Markdown Parsing and Section-Level Chunking

### Why Section-Level Chunking for Education?

Lecture notes have a clear **hierarchical structure**: topic → sub-topic → content.
Chunking at heading boundaries preserves this pedagogical hierarchy:

- A chunk about "Gradient Descent" sits under "Linear Models > Gradient Descent"
- The breadcrumb path lets the tutor cite the exact location in the notes
- Students can navigate back to the original section for deeper reading

**Chunking strategy:**
1. Split on `#`, `##`, `###` headings.
2. Keep each section's content as one chunk (typically 100–500 words).
3. Attach metadata: module, heading path (breadcrumb), tags, difficulty.
4. Discard chunks shorter than `MIN_CHUNK_LEN` (noise from empty headings).


In [ ]:
@dataclass
class LectureSection:
    """A single section extracted from lecture notes."""
    module_id: str
    module_title: str
    heading_path: str          # e.g. "Linear Models > Gradient Descent > Variants"
    content: str               # raw text of the section
    tags: List[str]            # module-level topic tags
    difficulty: str            # beginner / intermediate / advanced
    depth: int                 # heading level (1=H1, 2=H2, 3=H3)

    @property
    def search_text(self) -> str:
        """Text used for indexing: heading path + content."""
        return f"{self.heading_path}\n{self.content}"

    def __repr__(self):
        preview = self.content[:60].replace("\n", " ")
        return f"Section({self.heading_path} | {preview}…)"


def parse_lecture_notes(lecture: dict) -> List[LectureSection]:
    """Parse markdown lecture content into section-level chunks."""
    content = lecture["content"].strip()
    heading_re = re.compile(r"^(#{1,3})\s+(.+)$", re.MULTILINE)

    sections = []
    # Find all heading positions
    headings = [(m.start(), len(m.group(1)), m.group(2).strip()) for m in heading_re.finditer(content)]

    # Build heading stack for breadcrumb
    heading_stack = []  # list of (level, text)

    for i, (pos, level, title) in enumerate(headings):
        # Determine end of this section
        end = headings[i + 1][0] if i + 1 < len(headings) else len(content)
        section_text = content[pos:end].strip()

        # Remove the heading line itself from section content
        first_newline = section_text.find("\n")
        if first_newline > 0:
            body = section_text[first_newline:].strip()
        else:
            body = ""

        # Update heading stack for breadcrumb
        while heading_stack and heading_stack[-1][0] >= level:
            heading_stack.pop()
        heading_stack.append((level, title))
        breadcrumb = " > ".join(h[1] for h in heading_stack)

        if len(body) >= MIN_CHUNK_LEN:
            sections.append(LectureSection(
                module_id=lecture["module_id"],
                module_title=lecture["title"],
                heading_path=breadcrumb,
                content=body,
                tags=lecture["tags"],
                difficulty=lecture["difficulty"],
                depth=level,
            ))

    return sections


# Parse all lectures
ALL_SECTIONS: List[LectureSection] = []
for lec in LECTURE_NOTES:
    secs = parse_lecture_notes(lec)
    ALL_SECTIONS.extend(secs)
    print(f"  {lec['module_id']}: {len(secs)} sections")

print(f"\nTotal sections: {len(ALL_SECTIONS)}")
print(f"Sample section: {ALL_SECTIONS[3]}")


## Building the Retrieval Index

We support two retrieval backends:

1. **TF-IDF + cosine similarity** (always available, no GPU needed)
2. **Sentence-transformer embeddings** (if `sentence-transformers` is installed)

Both produce a ranked list of sections given a query string.


In [ ]:
class TFIDFIndex:
    """TF-IDF based retrieval over lecture sections."""

    def __init__(self, sections: List[LectureSection]):
        self.sections = sections
        self.vectorizer = TfidfVectorizer(
            stop_words="english",
            max_features=5000,
            ngram_range=(1, 2),
        )
        texts = [s.search_text for s in sections]
        self.tfidf_matrix = self.vectorizer.fit_transform(texts)
        print(f"TF-IDF index built: {self.tfidf_matrix.shape[0]} docs, "
              f"{self.tfidf_matrix.shape[1]} features")

    def search(self, query: str, top_k: int = TOP_K) -> List[Tuple[LectureSection, float]]:
        q_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(q_vec, self.tfidf_matrix).flatten()
        top_idx = scores.argsort()[::-1][:top_k]
        return [(self.sections[i], float(scores[i])) for i in top_idx if scores[i] > 0]


class EmbeddingIndex:
    """Sentence-transformer embedding retrieval (optional)."""

    def __init__(self, sections: List[LectureSection], model_name: str = EMBED_MODEL):
        from sentence_transformers import SentenceTransformer
        self.sections = sections
        self.model = SentenceTransformer(model_name)
        texts = [s.search_text for s in sections]
        self.embeddings = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
        print(f"Embedding index built: {self.embeddings.shape}")

    def search(self, query: str, top_k: int = TOP_K) -> List[Tuple[LectureSection, float]]:
        q_emb = self.model.encode([query], normalize_embeddings=True)
        scores = (self.embeddings @ q_emb.T).flatten()
        top_idx = scores.argsort()[::-1][:top_k]
        return [(self.sections[i], float(scores[i])) for i in top_idx if scores[i] > 0]


# Build the index
if USE_ST:
    index = EmbeddingIndex(ALL_SECTIONS)
    INDEX_TYPE = "embedding"
else:
    index = TFIDFIndex(ALL_SECTIONS)
    INDEX_TYPE = "tfidf"

print(f"\nActive index: {INDEX_TYPE}")


## Metadata-Boosted Retrieval

Raw similarity scores treat all sections equally. In educational settings,
we want to boost sections that are:

1. **Tag-relevant** — query mentions a topic tag from the module.
2. **Appropriate difficulty** — match the student's level.
3. **Top-level concepts** — prefer H1/H2 sections over deeply nested H3 sub-points
   (unless the query is very specific).

The boosted score is:

    final_score = raw_score + tag_boost + difficulty_boost + depth_boost


In [ ]:
def boosted_search(
    query: str,
    index,
    top_k: int = TOP_K,
    student_level: str = "intermediate",
    tag_boost: float = 0.05,
    level_boost: float = 0.03,
    depth_penalty: float = 0.02,
) -> List[Tuple[LectureSection, float, float]]:
    """Search with metadata boosting. Returns (section, raw_score, boosted_score)."""
    raw_results = index.search(query, top_k=top_k * 2)  # over-fetch then re-rank
    query_lower = query.lower()

    boosted = []
    for section, raw_score in raw_results:
        boost = 0.0
        # Tag match
        for tag in section.tags:
            if tag.replace("-", " ") in query_lower or tag.replace("-", "") in query_lower:
                boost += tag_boost
                break
        # Difficulty alignment
        if section.difficulty == student_level:
            boost += level_boost
        # Depth: prefer H1/H2 over H3 (lower depth = more general = slight boost)
        if section.depth <= 2:
            boost += depth_penalty
        boosted.append((section, raw_score, raw_score + boost))

    boosted.sort(key=lambda x: x[2], reverse=True)
    return boosted[:top_k]


# Quick test
test_results = boosted_search("What is gradient descent?", index)
print(f"Test query: 'What is gradient descent?'")
print(f"Top {len(test_results)} results:")
for sec, raw, boosted in test_results:
    print(f"  [{sec.module_id}] {sec.heading_path}")
    print(f"    raw={raw:.4f}  boosted={boosted:.4f}")


## Baseline: Keyword Search

Before evaluating the tutor, we establish a simple keyword-matching baseline.
This counts exact word overlaps between the query and each section — no
semantic understanding, no weighting.


In [ ]:
def keyword_search(query: str, sections: List[LectureSection], top_k: int = TOP_K):
    """Simple keyword overlap search."""
    query_words = set(query.lower().split())
    scored = []
    for sec in sections:
        text_words = set(sec.search_text.lower().split())
        overlap = len(query_words & text_words)
        if overlap > 0:
            scored.append((sec, overlap / len(query_words)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


# Compare
kw_results = keyword_search("What is gradient descent?", ALL_SECTIONS)
print("Keyword baseline top results:")
for sec, score in kw_results[:3]:
    print(f"  [{sec.module_id}] {sec.heading_path} — overlap={score:.2f}")


## Course Material Tutor Pipeline

The `CourseTutor` class wraps the retrieval engine and exposes three study modes:

| Mode | Input | Output |
|------|-------|--------|
| **Q&A** | A question about course content | Answer with citations |
| **Summary** | A topic name | Bullet-point summary from relevant sections |
| **Quiz** | A topic name | Multiple-choice question with answer key |

All outputs are **grounded** — they cite the exact section and module from which
the information was derived. No hallucinated answers.


In [ ]:
class CourseTutor:
    """Multi-mode study assistant over lecture notes."""

    def __init__(self, index, sections: List[LectureSection]):
        self.index = index
        self.sections = sections

    # ── Q&A Mode ──────────────────────────────────────────
    def ask(self, question: str, top_k: int = 3, student_level: str = "intermediate") -> dict:
        """Answer a question using retrieved lecture sections."""
        results = boosted_search(question, self.index, top_k=top_k, student_level=student_level)
        if not results:
            return {"answer": "No relevant content found in the lecture notes.", "sources": []}

        # Build answer from top sections
        answer_parts = []
        sources = []
        for sec, raw, boosted in results:
            # Extract the most relevant sentences (first ~3 sentences)
            sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', sec.content) if len(s.strip()) > 20]
            excerpt = " ".join(sentences[:3])
            answer_parts.append(excerpt)
            sources.append({
                "module": sec.module_id,
                "section": sec.heading_path,
                "score": round(boosted, 4),
            })

        combined = "\n\n".join(answer_parts)
        return {"answer": combined, "sources": sources}

    # ── Summary Mode ──────────────────────────────────────
    def summarise(self, topic: str, top_k: int = 5, student_level: str = "intermediate") -> dict:
        """Generate a bullet-point summary for a topic."""
        results = boosted_search(topic, self.index, top_k=top_k, student_level=student_level)
        if not results:
            return {"summary": "No content found for this topic.", "sources": []}

        bullets = []
        sources = []
        seen_headings = set()
        for sec, raw, boosted in results:
            if sec.heading_path in seen_headings:
                continue
            seen_headings.add(sec.heading_path)
            # Extract key facts as bullets
            lines = sec.content.strip().splitlines()
            key_lines = [l.strip() for l in lines
                         if l.strip() and not l.strip().startswith("|") and len(l.strip()) > 15][:3]
            for line in key_lines:
                clean = line.lstrip("- *•").strip()
                if clean and clean not in bullets:
                    bullets.append(clean)
            sources.append({
                "module": sec.module_id,
                "section": sec.heading_path,
            })

        summary_text = "\n".join(f"• {b}" for b in bullets[:10])
        return {"topic": topic, "summary": summary_text, "sources": sources}

    # ── Quiz Mode ─────────────────────────────────────────
    def quiz(self, topic: str, num_questions: int = 3, student_level: str = "intermediate") -> List[dict]:
        """Generate MCQ questions from lecture content on the given topic."""
        results = boosted_search(topic, self.index, top_k=num_questions * 2, student_level=student_level)
        if not results:
            return []

        questions = []
        used_sections = set()
        for sec, raw, boosted in results:
            if sec.heading_path in used_sections:
                continue
            used_sections.add(sec.heading_path)
            q = self._make_mcq(sec)
            if q:
                questions.append(q)
            if len(questions) >= num_questions:
                break
        return questions

    def _make_mcq(self, section: LectureSection) -> Optional[dict]:
        """Generate one MCQ from a section using extractive patterns."""
        content = section.content.strip()

        # Strategy 1: Definition-based — find "X is Y" patterns
        def_match = re.search(
            r"\*\*(.+?)\*\*[:\s]+(.+?)(?:\.|\n)",
            content
        )
        if def_match:
            term = def_match.group(1).strip()
            definition = def_match.group(2).strip().rstrip(".")
            # Generate distractors from other sections
            distractors = self._get_distractors(definition, section)
            if len(distractors) >= QUIZ_DISTRACTORS:
                options = [definition] + distractors[:QUIZ_DISTRACTORS]
                random.shuffle(options)
                return {
                    "question": f"What is {term}?",
                    "options": options,
                    "correct": definition,
                    "source": f"[{section.module_id}] {section.heading_path}",
                }

        # Strategy 2: List-based — if section has a bullet list,
        # ask which item belongs to this category
        bullets = re.findall(r"-\s+\*\*(.+?)\*\*", content)
        if len(bullets) >= 2:
            correct = random.choice(bullets)
            other_bullets = self._get_other_bullets(section)
            distractors = [b for b in other_bullets if b != correct][:QUIZ_DISTRACTORS]
            if len(distractors) >= QUIZ_DISTRACTORS:
                heading = section.heading_path.split(" > ")[-1]
                options = [correct] + distractors
                random.shuffle(options)
                return {
                    "question": f"Which of the following is related to {heading}?",
                    "options": options,
                    "correct": correct,
                    "source": f"[{section.module_id}] {section.heading_path}",
                }

        return None

    def _get_distractors(self, correct: str, exclude_section: LectureSection) -> List[str]:
        """Find plausible wrong answers from other sections."""
        distractors = []
        for sec in self.sections:
            if sec.heading_path == exclude_section.heading_path:
                continue
            defs = re.findall(r"\*\*(.+?)\*\*[:\s]+(.+?)(?:\.|\n)", sec.content)
            for term, defn in defs:
                clean = defn.strip().rstrip(".")
                if clean != correct and len(clean) > 10 and clean not in distractors:
                    distractors.append(clean)
            if len(distractors) >= QUIZ_DISTRACTORS * 2:
                break
        random.shuffle(distractors)
        return distractors

    def _get_other_bullets(self, exclude_section: LectureSection) -> List[str]:
        """Collect bold bullet items from other sections for distractors."""
        items = []
        for sec in self.sections:
            if sec.module_id == exclude_section.module_id:
                continue
            found = re.findall(r"-\s+\*\*(.+?)\*\*", sec.content)
            items.extend(found)
        random.shuffle(items)
        return items

    # ── Display helpers ───────────────────────────────────
    def display_answer(self, result: dict):
        print("=" * 70)
        print("ANSWER")
        print("=" * 70)
        print(textwrap.fill(result["answer"], width=70))
        print()
        print("Sources:")
        for s in result["sources"]:
            score_str = f" (score: {s['score']})" if "score" in s else ""
            print(f"  📖 [{s['module']}] {s['section']}{score_str}")
        print()

    def display_summary(self, result: dict):
        print("=" * 70)
        print(f"SUMMARY: {result['topic']}")
        print("=" * 70)
        print(result["summary"])
        print()
        print("Sources:")
        for s in result["sources"]:
            print(f"  📖 [{s['module']}] {s['section']}")
        print()

    def display_quiz(self, questions: List[dict]):
        print("=" * 70)
        print("QUIZ")
        print("=" * 70)
        for i, q in enumerate(questions, 1):
            print(f"\nQ{i}. {q['question']}")
            for j, opt in enumerate(q["options"]):
                marker = "✓" if opt == q["correct"] else " "
                print(f"  [{chr(65+j)}] {opt}  {marker}")
            print(f"  → Answer: {q['correct']}")
            print(f"  → Source: {q['source']}")
        print()


tutor = CourseTutor(index, ALL_SECTIONS)
print("Course Material Tutor initialised.")
print(f"Corpus: {len(ALL_SECTIONS)} sections from {len(LECTURE_NOTES)} modules")


## Q&A Mode — Demonstrations

In [ ]:
# Q1: Conceptual question
result = tutor.ask("What is the bias-variance tradeoff?")
tutor.display_answer(result)


In [ ]:
# Q2: Specific technique
result = tutor.ask("How does L1 regularisation differ from L2?")
tutor.display_answer(result)


In [ ]:
# Q3: Comparison
result = tutor.ask("What is the difference between bagging and boosting?")
tutor.display_answer(result)


In [ ]:
# Q4: Advanced topic
result = tutor.ask("Explain backpropagation and the vanishing gradient problem")
tutor.display_answer(result)


## Summary Mode — Demonstrations

In [ ]:
# Summarise a broad topic
result = tutor.summarise("ensemble methods")
tutor.display_summary(result)


In [ ]:
# Summarise a specific topic
result = tutor.summarise("classification metrics and evaluation")
tutor.display_summary(result)


## Quiz Mode — Demonstrations

In [ ]:
# Generate quiz on gradient descent
questions = tutor.quiz("gradient descent and optimisation", num_questions=3)
tutor.display_quiz(questions)


In [ ]:
# Generate quiz on classification
questions = tutor.quiz("classification and decision boundaries", num_questions=3)
tutor.display_quiz(questions)


## Evaluation

We evaluate the tutor with a hand-crafted eval set of 10 queries, each with
expected module and section breadcrumbs. Metrics:

- **Module Recall@1** — does the top result come from the correct module?
- **Section Recall@3** — does the correct section appear in top-3 results?
- **MRR (Mean Reciprocal Rank)** — average of 1/rank of the first correct result.


In [ ]:
EVAL_SET = [
    {"query": "What is machine learning?",
     "expected_module": "ML01", "expected_breadcrumb": "What is Machine Learning?"},
    {"query": "Explain supervised vs unsupervised learning",
     "expected_module": "ML01", "expected_breadcrumb": "Types of Machine Learning"},
    {"query": "How does gradient descent work?",
     "expected_module": "ML02", "expected_breadcrumb": "Gradient Descent"},
    {"query": "What is Ridge regression?",
     "expected_module": "ML02", "expected_breadcrumb": "Regularisation"},
    {"query": "Explain logistic regression",
     "expected_module": "ML03", "expected_breadcrumb": "Logistic Regression"},
    {"query": "What are precision and recall?",
     "expected_module": "ML03", "expected_breadcrumb": "Classification Metrics"},
    {"query": "How does random forest work?",
     "expected_module": "ML04", "expected_breadcrumb": "Random Forest"},
    {"query": "What is cross-validation?",
     "expected_module": "ML05", "expected_breadcrumb": "Cross-Validation"},
    {"query": "Explain the vanishing gradient problem",
     "expected_module": "ML06", "expected_breadcrumb": "Backpropagation"},
    {"query": "What activation functions are used in neural networks?",
     "expected_module": "ML06", "expected_breadcrumb": "Activation Functions"},
]

def evaluate_retrieval(search_fn, eval_set, label=""):
    """Evaluate retrieval quality."""
    module_recall_1 = 0
    section_recall_3 = 0
    mrr_sum = 0

    for item in eval_set:
        results = search_fn(item["query"])
        # Module Recall@1
        if results and results[0][0].module_id == item["expected_module"]:
            module_recall_1 += 1
        # Section Recall@3 and MRR
        found_rank = None
        for rank, (sec, *_) in enumerate(results[:3], 1):
            if item["expected_breadcrumb"].lower() in sec.heading_path.lower():
                if found_rank is None:
                    found_rank = rank
        if found_rank:
            section_recall_3 += 1
            mrr_sum += 1.0 / found_rank

    n = len(eval_set)
    metrics = {
        "Module Recall@1": module_recall_1 / n,
        "Section Recall@3": section_recall_3 / n,
        "MRR": mrr_sum / n,
    }
    print(f"\n{'=' * 50}")
    print(f"  Retrieval Evaluation: {label}")
    print(f"{'=' * 50}")
    for k, v in metrics.items():
        bar = "█" * int(v * 20) + "░" * (20 - int(v * 20))
        print(f"  {k:<20s}: {v:.2f}  {bar}")
    return metrics


# Boosted search wrapper
def boosted_wrapper(query):
    return [(sec, raw) for sec, raw, boosted in boosted_search(query, index)]

# Keyword baseline wrapper
def keyword_wrapper(query):
    return keyword_search(query, ALL_SECTIONS)

print("Evaluating retrieval backends …\n")
boosted_metrics  = evaluate_retrieval(boosted_wrapper, EVAL_SET, label=f"Boosted {INDEX_TYPE.upper()}")
keyword_metrics  = evaluate_retrieval(keyword_wrapper, EVAL_SET, label="Keyword Baseline")


## Retrieval Comparison

In [ ]:
print(f"{'Metric':<22s} {'Boosted':>10s} {'Keyword':>10s} {'Delta':>10s}")
print("-" * 55)
for metric in ["Module Recall@1", "Section Recall@3", "MRR"]:
    b = boosted_metrics[metric]
    k = keyword_metrics[metric]
    delta = b - k
    sign = "+" if delta >= 0 else ""
    print(f"{metric:<22s} {b:>10.2f} {k:>10.2f} {sign}{delta:>9.2f}")
print()

if boosted_metrics["MRR"] > keyword_metrics["MRR"]:
    print("✓ Boosted retrieval outperforms keyword baseline as expected.")
    print("  Metadata boosting + semantic similarity provides better ranking.")
else:
    print("△ Keyword baseline competitive — corpus may be small enough for exact match.")
    print("  With a larger corpus, embedding/TF-IDF retrieval will increasingly dominate.")


## Error Analysis

In [ ]:
# Test edge cases
edge_cases = [
    ("Out-of-scope: cooking", "How do I make pancakes?"),
    ("Ambiguous: tree", "Tell me about trees"),
    ("Very specific", "What learning rate should I use for Adam with batch_size=64?"),
    ("Cross-topic", "How do random forests relate to gradient descent?"),
    ("Misspelled", "What is bais-varience tradeoff?"),
]

print("Edge Case Analysis")
print("=" * 70)
for label, query in edge_cases:
    results = boosted_search(query, index, top_k=2)
    print(f"\n[{label}]")
    print(f"  Query: {query}")
    if results:
        top = results[0]
        print(f"  Top result: [{top[0].module_id}] {top[0].heading_path}")
        print(f"  Score: {top[2]:.4f}")
        relevant = top[2] > 0.10
        print(f"  Relevant: {'Yes' if relevant else 'Probably not — low score'}")
    else:
        print("  No results found.")


## Limitations

1. **No generative model** — The tutor extracts and reorganises text from the notes.
   It cannot paraphrase, explain in simpler words, or generate new examples.
   Adding an LLM (e.g., Qwen3-Instruct) for answer generation would fix this.

2. **Quiz quality is extractive** — MCQs are generated from bold-term definitions
   and bullet lists. Questions about reasoning, proofs, or multi-step procedures
   cannot be auto-generated without an LLM.

3. **Small corpus** — 6 modules / ~50 sections. TF-IDF works surprisingly well
   here because the vocabulary is small. With a real 30-lecture course,
   embedding-based retrieval would be significantly better.

4. **No student model** — The tutor doesn't track what the student already knows.
   A production system would maintain a knowledge state and adapt difficulty.

5. **No multi-turn dialogue** — Each query is independent. A conversational
   version would benefit from context history (e.g., "Tell me more about that").

6. **Keyword sensitivity** — TF-IDF retrieval relies on exact word matches.
   Typos, synonyms, and informal language reduce retrieval quality.


## Common Mistakes

| Mistake | Why it's wrong | Fix |
|---------|---------------|-----|
| Chunking at fixed character counts | Breaks mid-sentence, loses heading context | Chunk at heading boundaries |
| Indexing full documents (not sections) | Retrieves too much irrelevant content per hit | Section-level chunking |
| Ignoring metadata | Misses topic relevance signals | Add tag/difficulty/depth boosting |
| Using quiz answers not from the actual notes | Hallucination risk | Extractive-only quiz generation |
| Evaluating only on easy queries | Overstates system quality | Include edge cases and adversarial queries |
| Skipping a baseline | Can't tell if the fancy retrieval actually helps | Always compare vs keyword |
| Mixing difficulty levels | Confuses introductory students with advanced content | Student-level parameter |


## Mini Challenge

Try these exercises to extend the tutor:

1. **Add a new module** — Write a 7th lecture on "Clustering" (K-Means, DBSCAN,
   silhouette score) and add it to `LECTURE_NOTES`. Re-build the index and test.

2. **Implement "Explain like I'm 5" mode** — When the student asks for a simple
   explanation, retrieve the relevant section and filter/simplify to the first
   sentence of each paragraph only.

3. **Add fill-in-the-blank quizzes** — Instead of MCQ, generate questions like
   "__________ is an optimisation algorithm that updates weights iteratively"
   by masking bold terms in the notes.

4. **Track student performance** — After each quiz, record correct/incorrect
   and implement spaced repetition (show topics the student got wrong more often).

5. **Compare TF-IDF vs BM25** — Implement BM25 scoring and evaluate whether it
   outperforms TF-IDF on the same eval set.


## Production Considerations

| Consideration | Approach |
|--------------|----------|
| **Corpus size** | Switch to embedding retrieval + vector DB (ChromaDB, FAISS) for 100+ lectures |
| **Answer quality** | Add Qwen3-Instruct or GPT-4o for generative answers with retrieval context |
| **Real-time updates** | Incremental indexing when new slides are uploaded |
| **Multi-format input** | Parse `.pptx` (python-pptx), `.pdf` (pymupdf), `.docx` (python-docx) into markdown |
| **Student auth** | Track per-student progress, quiz scores, weak topics for adaptive learning |
| **Quiz validation** | Human review pipeline for auto-generated questions before student-facing use |
| **Multi-language** | Use multilingual embeddings (BGE-M3) for courses in other languages |
| **Accessibility** | TTS for answers, screen-reader-friendly output formatting |
| **LMS integration** | Export quizzes as QTI/SCORM for Moodle/Canvas/Blackboard import |


## How to Improve This Project

1. **Add an LLM for answer generation** — Use the retrieved sections as context
   for a language model (Qwen3-Instruct, Llama 3.1) to produce fluent, paraphrased
   answers instead of raw text extraction.

2. **Hybrid retrieval** — Combine TF-IDF (exact matches) with embedding similarity
   (semantic matches) using reciprocal rank fusion (RRF).

3. **Cross-reference detection** — When answering about regularisation, also
   surface related sections on overfitting and bias-variance for prerequisite context.

4. **Adaptive difficulty** — Start with beginner-level content; if the student
   answers quiz questions correctly, progressively surface intermediate/advanced sections.

5. **Multi-hop Q&A** — Some questions span multiple modules ("How do ensemble
   methods help with the bias-variance tradeoff?"). Chain retrieval across modules.

6. **PDF/PPTX ingestion** — Replace the embedded corpus with a file-loader that
   parses real uploaded course files into the same section format.


## Key Takeaways

1. **Educational RAG needs heading-aware chunking** — lecture notes have clear
   topic-subtopic structure; respecting it produces far better retrieval than
   fixed-size chunks.

2. **Metadata boosting matters** — topic tags, difficulty level, and heading depth
   are cheap signals that meaningfully improve ranking quality.

3. **Three modes, one index** — Q&A, summary, and quiz all use the same retrieval
   engine with different post-processing. The retrieval layer is the core investment.

4. **Extractive quiz generation works** — bold-term definitions and bullet lists
   are reliable sources for auto-generated MCQs, even without an LLM.

5. **Always compare against a keyword baseline** — it's fast to implement and
   proves whether your retrieval system actually adds value.

6. **TF-IDF is viable for small corpora** — with 50 sections and domain-specific
   vocabulary, TF-IDF performs well. Switch to embeddings when the corpus grows
   beyond ~200 sections.

7. **Production path is clear** — add an LLM for generation, a vector DB for
   scale, and a student model for personalisation. The architecture is modular.
